In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn.functional as F
from PIL import Image
import os
from tqdm import tqdm
import time
import gc
import glob
import warnings
warnings.filterwarnings('ignore')

# Проверка версий
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Оптимизации для GPU
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ФУНКЦИИ ДЛЯ УПРАВЛЕНИЯ ПАМЯТЬЮ
def clean_memory():
    """Очистка памяти GPU"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def print_gpu_memory():
    """Вывод информации о памяти GPU"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory: {allocated:.2f}/{total:.2f} GB")

In [ ]:
# АРХИТЕКТУРА SIMSIAM (Chen et al., "Exploring Simple Siamese Representation Learning", 2021)
class SimSiam(nn.Module):
    """
    SimSiam: базовая архитектура без модификаций
    """
    def __init__(self, backbone='resnet50', dim=2048, pred_dim=512):
        super(SimSiam, self).__init__()

        # Encoder: ResNet50
        self.encoder = models.resnet50(weights=None)

        # Projection MLP
        self.encoder.fc = nn.Sequential(
            nn.Linear(2048, 2048, bias=False),
            nn.BatchNorm1d(2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048, dim, bias=False),
            nn.BatchNorm1d(dim),
            nn.ReLU(inplace=True),
            nn.Linear(dim, dim, bias=False),
            nn.BatchNorm1d(dim, affine=False)
        )

        # Prediction MLP
        self.predictor = nn.Sequential(
            nn.Linear(dim, pred_dim, bias=False),
            nn.BatchNorm1d(pred_dim),
            nn.ReLU(inplace=True),
            nn.Linear(pred_dim, dim)
        )

    def forward(self, x1, x2):
        z1 = self.encoder(x1)
        z2 = self.encoder(x2)

        p1 = self.predictor(z1)
        p2 = self.predictor(z2)

        # stop-gradient оператор
        return p1, p2, z1.detach(), z2.detach()

In [ ]:
# ФУНКЦИИ ПОТЕРЬ
def negative_cosine_similarity(p, z):
    """Отрицательное косинусное сходство"""
    p = F.normalize(p, dim=1)
    z = F.normalize(z, dim=1)
    return -(p * z).sum(dim=1).mean()

def simsiam_loss(p1, p2, z1, z2):
    """Симметричная функция потерь SimSiam"""
    loss = 0.5 * (negative_cosine_similarity(p1, z2) +
                  negative_cosine_similarity(p2, z1))
    return loss

In [ ]:
# БАЗОВЫЙ ДАТАСЕТ ДЛЯ ЦИТОЛОГИЧЕСКИХ ИЗОБРАЖЕНИЙ
class CytologyDataset(Dataset):
    """
    Базовый датасет для цитологических изображений

    Особенности:
    - Нарезка изображений на патчи 512x512 по сетке 5x5
    - Ресайз до 224x224 для ResNet50
    - Стандартные аугментации SimSiam
    - НЕТ фильтрации пустых патчей (baseline)
    """
    def __init__(self, data_dir, patch_size=512, target_size=224):
        self.data_dir = data_dir
        self.patch_size = patch_size
        self.target_size = target_size
        self.patches_per_image = 25  # сетка 5x5

        # Поиск всех изображений
        self.image_paths = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']:
            self.image_paths.extend(glob.glob(
                os.path.join(data_dir, '**', ext), recursive=True))

        print(f"Найдено изображений: {len(self.image_paths)}")
        print(f"Всего патчей: {len(self.image_paths) * self.patches_per_image}")

        # БАЗОВЫЕ АУГМЕНТАЦИИ (как в оригинальном SimSiam)
        self.transform = transforms.Compose([
            # RandomResizedCrop - стандартный для SimSiam
            transforms.RandomResizedCrop(
                target_size,
                scale=(0.2, 1.0)
            ),

            # Базовые пространственные аугментации
            transforms.RandomHorizontalFlip(p=0.5),

            # Базовые цветовые аугментации
            transforms.RandomApply([
                transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
            ], p=0.8),

            transforms.RandomGrayscale(p=0.2),

            # Gaussian blur (как в SimSiam)
            transforms.GaussianBlur(
                kernel_size=3,
                sigma=(0.1, 2.0)
            ),

            # В тензор и нормализация ImageNet
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.image_paths) * self.patches_per_image

    def _extract_patch(self, img, idx):
        """Вырезание патча по сетке 5x5"""
        width, height = img.size
        patch_idx = idx % self.patches_per_image

        # Сетка 5x5
        rows = cols = 5

        # Шаг между патчами
        stride_h = max(1, (height - self.patch_size) // (rows - 1)) if rows > 1 else 0
        stride_w = max(1, (width - self.patch_size) // (cols - 1)) if cols > 1 else 0

        # Позиция патча
        row = min((patch_idx // cols) * stride_h, height - self.patch_size)
        col = min((patch_idx % cols) * stride_w, width - self.patch_size)

        # Вырезаем
        patch = img.crop((col, row, col + self.patch_size, row + self.patch_size))

        return patch

    def __getitem__(self, idx):
        # Индекс изображения
        img_idx = idx // self.patches_per_image
        img_idx = min(img_idx, len(self.image_paths) - 1)

        img_path = self.image_paths[img_idx]

        try:
            with Image.open(img_path) as img:
                img = img.convert('RGB')

                # Вырезаем патч 512x512
                patch = self._extract_patch(img, idx)

                # Два разных view
                view1 = self.transform(patch)
                view2 = self.transform(patch)

                return view1, view2

        except Exception as e:
            print(f"Ошибка загрузки {os.path.basename(img_path)}: {e}")
            dummy = torch.zeros(3, self.target_size, self.target_size)
            return dummy, dummy

In [ ]:
# ФУНКЦИИ ДЛЯ РАБОТЫ С ЧЕКПОИНТАМИ
def save_checkpoint(epoch, model, optimizer, scheduler, loss, save_dir, is_best=False):
    """Сохранение чекпоинта"""
    os.makedirs(save_dir, exist_ok=True)

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'loss': loss,
    }

    # Сохраняем
    path = os.path.join(save_dir, f'checkpoint_epoch_{epoch:03d}.pth')
    torch.save(checkpoint, path)

    if is_best:
        best_path = os.path.join(save_dir, 'best_model.pth')
        torch.save(checkpoint, best_path)
        print(f" Сохранена лучшая модель! Loss: {loss:.4f}")

    print(f" Сохранено: {path}")

def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None):
    """Загрузка чекпоинта"""
    if not os.path.exists(checkpoint_path):
        print(f"Чекпоинт не найден: {checkpoint_path}")
        return 0, float('inf')

    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if scheduler and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    epoch = checkpoint.get('epoch', 0)
    loss = checkpoint.get('loss', float('inf'))

    print(f"Загружен чекпоинт эпохи {epoch}, loss: {loss:.4f}")
    return epoch, loss

In [ ]:
# ОСНОВНАЯ ФУНКЦИЯ ОБУЧЕНИЯ
def main():
    # ПАРАМЕТРЫ
    DATA_DIR = "/content/ALL_JPEGImages"
    SAVE_DIR = "/content/drive/MyDrive/checkpoints/simsiam_baseline"
    BATCH_SIZE = 512
    PATCH_SIZE = 512
    TARGET_SIZE = 224
    EPOCHS = 100
    NUM_WORKERS = 8
    LEARNING_RATE = 0.05 * BATCH_SIZE / 256  # linear scaling

    print(f"Параметры:")
    print(f"  - Batch size: {BATCH_SIZE}")
    print(f"  - Patch size: {PATCH_SIZE}×{PATCH_SIZE}")
    print(f"  - Target size: {TARGET_SIZE}×{TARGET_SIZE}")
    print(f"  - Эпох: {EPOCHS}")
    print(f"  - Learning rate: {LEARNING_RATE:.6f}")

    # Устройство
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nУстройство: {device}")

    # Датасет
    print("\nЗагрузка данных...")
    dataset = CytologyDataset(
        data_dir=DATA_DIR,
        patch_size=PATCH_SIZE,
        target_size=TARGET_SIZE
    )

    if len(dataset) == 0:
        print("Нет данных для обучения!")
        return

    # DataLoader
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
        drop_last=True
    )

    print(f"\nБатчей в эпоху: {len(dataloader)}")

    # Модель
    print("\nСоздание модели...")
    model = SimSiam().to(device)

    # Оптимизатор
    optimizer = optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=0.9,
        weight_decay=1e-4,
        nesterov=True
    )

    # Scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    # Обучение
    best_loss = float('inf')
    loss_history = []

    for epoch in range(EPOCHS):
        # Очистка памяти
        clean_memory()

        model.train()
        epoch_loss = 0
        epoch_start = time.time()

        progress_bar = tqdm(dataloader, desc=f'Эпоха {epoch+1}/{EPOCHS}')

        for batch_idx, (x1, x2) in enumerate(progress_bar):
            # На GPU
            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)

            # Forward
            optimizer.zero_grad(set_to_none=True)
            p1, p2, z1, z2 = model(x1, x2)
            loss = simsiam_loss(p1, p2, z1, z2)

            # Backward
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            # Прогресс
            if batch_idx % 10 == 0:
                avg_loss = epoch_loss / (batch_idx + 1)
                progress_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'avg': f'{avg_loss:.4f}'
                })

        # Статистика эпохи
        epoch_time = time.time() - epoch_start
        avg_epoch_loss = epoch_loss / len(dataloader)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        loss_history.append(avg_epoch_loss)

        print(f"\nЭпоха {epoch+1}: loss = {avg_epoch_loss:.4f}, "
              f"время = {epoch_time:.1f}с, lr = {current_lr:.6f}")

        # Сохраняем каждые 10 эпох
        if (epoch + 1) % 10 == 0:
            save_checkpoint(epoch + 1, model, optimizer, scheduler,
                          avg_epoch_loss, SAVE_DIR)

        # Лучшая модель
        if avg_epoch_loss < best_loss:
            best_loss = avg_epoch_loss
            save_checkpoint(epoch + 1, model, optimizer, scheduler,
                          avg_epoch_loss, SAVE_DIR, is_best=True)

    print(f"Лучший loss: {best_loss:.4f}")
    print(f"Чекпоинты сохранены в: {SAVE_DIR}")

    return model, loss_history

In [ ]:
# ЗАПУСК ОБУЧЕНИЯ
if __name__ == "__main__":
    # Очистка памяти перед запуском
    clean_memory()

    # Запуск обучения
    model, losses = main()